In [30]:
import pandas as pd
import numpy as np
import plotly.express as px

In [31]:
pip install feature-engine

Note: you may need to restart the kernel to use updated packages.


In [32]:
wine=pd.read_csv('/kaggle/input/Tutorials/Outliers/winequality-red.csv')
wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [33]:
wine.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [34]:
def make_plot(df, colors):
    for feature in df.columns:
        fig=px.box(df,y=feature,title="<b>"+feature,color=colors,boxmode="group",template="plotly_dark",points='all')
        fig.update_layout(title_x=0.5,title_font_size=30,font_size=15,font_color='aqua')
        fig.show()
      

In [35]:
make_plot(wine,wine['quality'])

In [36]:
X=wine.drop(columns='quality')
y=wine['quality']

In [37]:
num_colums=wine.columns.to_list()
num_colums.pop()

'quality'

In [38]:
def find_boundaries(df,feature,distance):
    IQR=df[feature].quantile(0.75) -df[feature].quantile(0.25)
    lower_boundary=df[feature].quantile(0.25)-(IQR*distance)
    upper_boundary=df[feature].quantile(0.75)+(IQR*distance)
    return upper_boundary,lower_boundary

In [39]:
for feature in num_colums:
    upper_boundary,lower_boundary=find_boundaries(wine,feature,1.5)
    print(feature,"upper->",upper_boundary.round(4),"lower->",lower_boundary.round(4),"\n....................")


fixed acidity upper-> 12.35 lower-> 3.95 
....................
volatile acidity upper-> 1.015 lower-> 0.015 
....................
citric acid upper-> 0.915 lower-> -0.405 
....................
residual sugar upper-> 3.65 lower-> 0.85 
....................
chlorides upper-> 0.12 lower-> 0.04 
....................
free sulfur dioxide upper-> 42.0 lower-> -14.0 
....................
total sulfur dioxide upper-> 122.0 lower-> -38.0 
....................
density upper-> 1.0012 lower-> 0.9922 
....................
pH upper-> 3.685 lower-> 2.925 
....................
sulphates upper-> 1.0 lower-> 0.28 
....................
alcohol upper-> 13.5 lower-> 7.1 
....................


#### Outline Trimmer
1. Gaussian Approximation( mean plus or minus three times the standard deviation)
2. IQR Rule
3. Percentiles( 5th and 95th quantiles)

In [40]:
from feature_engine.outliers import OutlierTrimmer

In [41]:
trimmer=OutlierTrimmer(capping_method='iqr',tail='both',fold=1.5,variables=num_colums)
wine_copy=wine.copy(deep=True)


In [42]:
trimmer.fit(wine_copy)

OutlierTrimmer(capping_method='iqr', fold=1.5, tail='both',
               variables=['fixed acidity', 'volatile acidity', 'citric acid',
                          'residual sugar', 'chlorides', 'free sulfur dioxide',
                          'total sulfur dioxide', 'density', 'pH', 'sulphates',
                          'alcohol'])

In [43]:
trimmer.left_tail_caps_

{'fixed acidity': 3.95,
 'volatile acidity': 0.015000000000000013,
 'citric acid': -0.4049999999999999,
 'residual sugar': 0.8499999999999996,
 'chlorides': 0.04000000000000002,
 'free sulfur dioxide': -14.0,
 'total sulfur dioxide': -38.0,
 'density': 0.9922475000000001,
 'pH': 2.925,
 'sulphates': 0.28000000000000014,
 'alcohol': 7.1000000000000005}

In [44]:
trimmer.right_tail_caps_

{'fixed acidity': 12.349999999999998,
 'volatile acidity': 1.0150000000000001,
 'citric acid': 0.9149999999999999,
 'residual sugar': 3.6500000000000004,
 'chlorides': 0.11999999999999998,
 'free sulfur dioxide': 42.0,
 'total sulfur dioxide': 122.0,
 'density': 1.0011875,
 'pH': 3.6849999999999996,
 'sulphates': 0.9999999999999999,
 'alcohol': 13.5}

In [45]:
wine_copy=trimmer.transform(wine_copy)

In [46]:
make_plot(wine_copy,wine_copy['quality'])

#### Winsorization(Capping)
* Gaussian limits:

right tail: mean + 3* std

left tail: mean - 3* std

* IQR limits:

right tail: 75th quantile + 3* IQR

left tail: 25th quantile - 3* IQR

where IQR is the inter-quartile range: 75th quantile - 25th quantile.

* percentiles or quantiles:

right tail: 95th percentile

left tail: 5th percentile

In [47]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(wine.drop(columns=['quality']),wine['quality'],random_state=42)


In [48]:
X_train_copy=X_train.copy()

In [49]:
from feature_engine.outliers import Winsorizer

In [50]:
winsor_trimmer=Winsorizer(capping_method='iqr',tail='both',fold=1.5)



In [51]:
winsor_trimmer.fit(X_train_copy)

Winsorizer(capping_method='iqr', fold=1.5, tail='both')

In [52]:
winsor_trimmer.right_tail_caps_

{'fixed acidity': 12.349999999999998,
 'volatile acidity': 1.0,
 'citric acid': 0.9249999999999999,
 'residual sugar': 3.6500000000000004,
 'chlorides': 0.12249999999999998,
 'free sulfur dioxide': 42.0,
 'total sulfur dioxide': 124.5,
 'density': 1.001085,
 'pH': 3.6849999999999996,
 'sulphates': 0.9999999999999999,
 'alcohol': 13.25}

In [55]:
X_test_copy=X_test.copy()

In [56]:
X_train_copy=winsor_trimmer.transform(X_train_copy)
X_test_copy=winsor_trimmer.transform(X_test_copy)